# ARIMA Data Import Validation Log

Purpose: verify that ARIMA data pulled locally into this project's `data/`
cache was imported correctly (not garbled, not silently truncated, encoding
intact) — a data-quality log, not an analysis notebook.

This does **not** re-derive or re-argue any conclusions from
`outputs/part1_vintage_comparison.md`. It only checks that the local copies
match what the earlier GCS/schema inspection (this session) already found.
Any "expected" value referenced below comes from that earlier inspection,
not from assumption.

Scope note: some sources are too large to fully cache locally without
violating the "don't download the entire dataset unnecessarily" project
convention. Where that applies (`intact-survey`, 250 shards / ~32.4M rows),
this notebook validates a **representative local sample** (first 3 shards +
the last shard) for schema/content checks, and cross-checks the *full*
row count via a lightweight remote metadata scan (reads only parquet
footers, not row data) rather than downloading all 250 shards.

No fixing or cleaning is done here — findings are reported, not corrected.


In [5]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

DATA = "/Users/wiame.ichane/arima_v24_v26/data"


## Helper functions

Shared checks reused across every section below, so each section reads the
same way.

In [6]:
def report_basic(df, name):
    """Shape, dtypes, memory footprint."""
    print(f"--- {name} ---")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]:,} cols")
    mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
    print(f"memory footprint: {mem_mb:,.1f} MB")
    dtype_counts = df.dtypes.value_counts()
    print("dtype counts:")
    for dt, n in dtype_counts.items():
        print(f"  {dt}: {n}")
    return mem_mb


def flag_null_or_constant(df, max_list=25):
    """Flag columns that are entirely null, or have exactly one distinct
    non-null value (constant) -- both are red flags for a botched import."""
    null_only = []
    constant = []
    for c in df.columns:
        s = df[c]
        if s.isna().all():
            null_only.append(c)
        else:
            nun = s.nunique(dropna=True)
            if nun == 1:
                constant.append(c)
    print(f"null-only columns: {len(null_only)}"
          + (f" -> {null_only[:max_list]}" if null_only else ""))
    print(f"constant (single-value) columns: {len(constant)}"
          + (f" -> {constant[:max_list]}" if constant else ""))
    return null_only, constant


def show_sample_decoded(df, cols, n=5):
    """Print a small sample of values for representative columns, to
    visually confirm the data looks sane (not garbled/truncated)."""
    cols = [c for c in cols if c in df.columns]
    print(f"sample values ({n} rows) for {cols}:")
    with pd.option_context("display.max_colwidth", 60):
        print(df[cols].head(n).to_string())


def check_row_count(actual, expected, label):
    status = "OK" if actual == expected else "MISMATCH"
    print(f"[{status}] row count — {label}: actual={actual:,} expected={expected:,}")
    return status


def check_id_range(id_series, label, expect_min=None, expect_max=None,
                    expect_gapfree=None, expect_nunique=None):
    """Regression check against today's finding: ARIMA id columns are
    dense, gap-free, sequential integer ranges (per export)."""
    n = len(id_series)
    nun = id_series.nunique()
    mn, mx = id_series.min(), id_series.max()
    print(f"id check — {label}: n={n:,} nunique={nun:,} min={mn} max={mx}")

    ok = True
    if expect_nunique is not None:
        s = "OK" if nun == expect_nunique else "MISMATCH"
        print(f"  [{s}] nunique expected={expect_nunique:,}")
        ok &= (s == "OK")
    if expect_min is not None:
        s = "OK" if mn == expect_min else "MISMATCH"
        print(f"  [{s}] min expected={expect_min}")
        ok &= (s == "OK")
    if expect_max is not None:
        s = "OK" if mx == expect_max else "MISMATCH"
        print(f"  [{s}] max expected={expect_max}")
        ok &= (s == "OK")
    if expect_gapfree:
        gapfree = (nun == (mx - mn + 1))
        s = "OK" if gapfree else "MISMATCH"
        print(f"  [{s}] gap-free contiguous range: {gapfree}")
        ok &= (s == "OK")
    return ok


## 1. `arima_200k_all417_raw.parquet` (2024, superseded `arima-clustering-pipeline` anchor)

Expected (from earlier GCS inspection this session): 200,000 rows x 11,879
cols, `id` unique (200,000), min=284, max=32,433,836, no nulls in `id`.

In [7]:
path = f"{DATA}/all417/arima_200k_all417_raw.parquet"
df = pd.read_parquet(path)
mem = report_basic(df, "all417 raw")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["id", "VV_DEM_1", "VV_DEM_2", "VV_ACN_1"])
check_row_count(df.shape[0], 200000, "all417 raw rows")
print(f"col count check: actual={df.shape[1]} expected=11879 -> "
      f"{'OK' if df.shape[1] == 11879 else 'MISMATCH'}")
check_id_range(df["id"], "all417 raw", expect_min=284, expect_max=32433836,
               expect_nunique=200000)
print(f"id nulls: {df['id'].isna().sum()}")


--- all417 raw ---
shape: 200,000 rows x 11,879 cols
memory footprint: 18,125.9 MB
dtype counts:
  int64: 11879
null-only columns: 0
constant (single-value) columns: 74 -> ['VV_BUS_57', 'VV_COS_13', 'VV_COS_25', 'VV_DAL_46', 'VV_DAL_51', 'VV_DAL1_28', 'VV_DAL1_33', 'VV_DAL1_55', 'VV_DAL1_60', 'VV_DAL10_21', 'VV_DAL10_26', 'VV_DAL10_53', 'VV_DAL11_7', 'VV_DAL11_83', 'VV_DAL11_88', 'VV_DAL12_10', 'VV_DAL12_15', 'VV_DAL2_14', 'VV_DAL2_36', 'VV_DAL2_41', 'VV_DAL2_90', 'VV_DAL2_95', 'VV_DAL3_22', 'VV_DAL3_49', 'VV_DAL4_31']
sample values (5 rows) for ['id', 'VV_DEM_1', 'VV_DEM_2', 'VV_ACN_1']:
     id  VV_DEM_1  VV_DEM_2  VV_ACN_1
0   284  21440004  21440014    650003
1   367  21440004  21440013    650003
2   431  21440010  21440013    650003
3  1390  21440012  21440014    650003
4  1432  21440004  21440014    650003
[OK] row count — all417 raw rows: actual=200,000 expected=200,000
col count check: actual=11879 expected=11879 -> OK
id check — all417 raw: n=200,000 nunique=200,000 min=284 ma

## 2. `arima_200k_all417_decoded.parquet`

Same expectations as raw (same `id`, decoded value labels instead of raw
VALUE_ID codes).

In [8]:
path = f"{DATA}/all417/arima_200k_all417_decoded.parquet"
df = pd.read_parquet(path)
mem = report_basic(df, "all417 decoded")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["id", "VV_DEM_1", "VV_DEM_2", "VV_ACN_1"])
check_row_count(df.shape[0], 200000, "all417 decoded rows")
print(f"col count check: actual={df.shape[1]} expected=11879 -> "
      f"{'OK' if df.shape[1] == 11879 else 'MISMATCH'}")
check_id_range(df["id"], "all417 decoded", expect_min=284, expect_max=32433836,
               expect_nunique=200000)

# raw vs decoded id sequences should be identical (same underlying rows)
raw_ids = pd.read_parquet(f"{DATA}/all417/arima_200k_all417_raw.parquet", columns=["id"])["id"]
same_ids = (raw_ids.values == df["id"].values).all()
print(f"[{'OK' if same_ids else 'MISMATCH'}] raw.id == decoded.id (positional): {same_ids}")

# decoded values should be human-readable strings, not raw codes
print(f"VV_DEM_1 dtype: {df['VV_DEM_1'].dtype}, sample uniques: {df['VV_DEM_1'].dropna().unique()[:5]}")
looks_decoded = df["VV_DEM_1"].dropna().astype(str).str.contains("-").any()
print(f"[{'OK' if looks_decoded else 'CHECK'}] VV_DEM_1 looks like decoded age-band text (contains '-')")


--- all417 decoded ---
shape: 200,000 rows x 11,879 cols
memory footprint: 25,718.9 MB
dtype counts:
  str: 11878
  int64: 1
null-only columns: 0
constant (single-value) columns: 74 -> ['VV_BUS_57', 'VV_COS_13', 'VV_COS_25', 'VV_DAL_46', 'VV_DAL_51', 'VV_DAL1_28', 'VV_DAL1_33', 'VV_DAL1_55', 'VV_DAL1_60', 'VV_DAL10_21', 'VV_DAL10_26', 'VV_DAL10_53', 'VV_DAL11_7', 'VV_DAL11_83', 'VV_DAL11_88', 'VV_DAL12_10', 'VV_DAL12_15', 'VV_DAL2_14', 'VV_DAL2_36', 'VV_DAL2_41', 'VV_DAL2_90', 'VV_DAL2_95', 'VV_DAL3_22', 'VV_DAL3_49', 'VV_DAL4_31']
sample values (5 rows) for ['id', 'VV_DEM_1', 'VV_DEM_2', 'VV_ACN_1']:
     id VV_DEM_1 VV_DEM_2 VV_ACN_1
0   284  30 - 34   Female       No
1   367  30 - 34     Male       No
2   431  60 - 64     Male       No
3  1390      70+   Female       No
4  1432  30 - 34   Female       No
[OK] row count — all417 decoded rows: actual=200,000 expected=200,000
col count check: actual=11879 expected=11879 -> OK
id check — all417 decoded: n=200,000 nunique=200,000 min=284

## 3. `arima_200k_all417_manifest.csv`

This is a column-level data dictionary (one row per ARIMA column), not
person-level data. Expected: 11,878 data rows (11,879 lines including
header, per earlier `wc -l` check), columns
`column, n_unique, null_pct, sample_decoded`.

In [9]:
path = f"{DATA}/all417/arima_200k_all417_manifest.csv"
df = pd.read_csv(path)
mem = report_basic(df, "all417 manifest")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["column", "n_unique", "null_pct", "sample_decoded"])
check_row_count(df.shape[0], 11878, "all417 manifest data rows")

# known gap: VV_DEM_* columns exist in the decoded/raw parquet files but were
# not found in this manifest earlier this session -- re-confirm that gap here.
vv_dem_in_manifest = df["column"].astype(str).str.startswith("VV_DEM_").sum()
print(f"VV_DEM_* rows found in manifest: {vv_dem_in_manifest} "
      f"(earlier finding: 0 -- a documentation gap, not a data problem)")


--- all417 manifest ---
shape: 11,878 rows x 4 cols
memory footprint: 0.7 MB
dtype counts:
  str: 2
  int64: 1
  float64: 1
null-only columns: 0
constant (single-value) columns: 1 -> ['null_pct']
sample values (5 rows) for ['column', 'n_unique', 'null_pct', 'sample_decoded']:
     column  n_unique  null_pct       sample_decoded
0  VV_ACN_1         3       0.0   ['No', 'No', 'No']
1  VV_ADB_1         3       0.0  ['No', 'Yes', 'No']
2  VV_ADB_2         2       0.0  ['No', 'No', 'Yes']
3  VV_ADB_3         2       0.0   ['No', 'No', 'No']
4  VV_ADB_4         2       0.0   ['No', 'No', 'No']
[OK] row count — all417 manifest data rows: actual=11,878 expected=11,878
VV_DEM_* rows found in manifest: 20 (earlier finding: 0 -- a documentation gap, not a data problem)


## 4. `CA_2024H2/VV_ACN` (2024H2 `arrima-snowflake`, 16 shards)

Expected (from earlier GCS inspection): 32,433,918 total rows across 16
shards, `id` a dense gap-free range 1..32,433,918, columns
`ID, GEO, VV_ACN_1`.

In [10]:
import glob
shard_paths = sorted(glob.glob(f"{DATA}/CA_2024H2/VV_ACN/*.parquet"))
print(f"local shards found: {len(shard_paths)} (expected 16)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df = pd.concat(dfs, ignore_index=True)
mem = report_basic(df, "CA_2024H2 VV_ACN (all shards concatenated)")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["ID", "GEO", "VV_ACN_1"])
check_row_count(df.shape[0], 32433918, "VV_ACN total rows (16 shards)")
check_id_range(df["ID"].astype("int64"), "VV_ACN (2024H2)",
               expect_min=1, expect_max=32433918, expect_gapfree=True,
               expect_nunique=32433918)
# GEO should look like Canadian postal codes (6 chars, letter-digit-letter...)
sample_geo = df["GEO"].dropna().head(5).tolist()
looks_postal = all(len(str(g)) == 6 for g in sample_geo)
print(f"[{'OK' if looks_postal else 'CHECK'}] GEO sample looks like 6-char postal codes: {sample_geo}")


local shards found: 16 (expected 16)
--- CA_2024H2 VV_ACN (all shards concatenated) ---
shape: 32,433,918 rows x 3 cols
memory footprint: 8,351.5 MB
dtype counts:
  object: 2
  str: 1
null-only columns: 0
constant (single-value) columns: 0
sample values (5 rows) for ['ID', 'GEO', 'VV_ACN_1']:
         ID     GEO VV_ACN_1
0  17846273  M5A2N8   650003
1  17846274  M5A2N8   650003
2  17846275  M5A2N8   650003
3  17846276  M5A2N8   650003
4  17846277  M5A2N8   650003
[OK] row count — VV_ACN total rows (16 shards): actual=32,433,918 expected=32,433,918
id check — VV_ACN (2024H2): n=32,433,918 nunique=32,433,918 min=1 max=32433918
  [OK] nunique expected=32,433,918
  [OK] min expected=1
  [OK] max expected=32433918
  [OK] gap-free contiguous range: True
[OK] GEO sample looks like 6-char postal codes: ['M5A2N8', 'M5A2N8', 'M5A2N8', 'M5A2N8', 'M5A2N8']


## 5. `CA_2024H2/LOC` (geography dimension table, 5 shards)

Not person-level data — one row per geographic area. Earlier inspection
only checked shard 0 (352,256 rows); this is the first time all 5 shards
are combined, so the total is a new observation, not a regression check
against a prior total.

In [11]:
shard_paths = sorted(glob.glob(f"{DATA}/CA_2024H2/LOC/*.parquet"))
print(f"local shards found: {len(shard_paths)} (expected 5)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df = pd.concat(dfs, ignore_index=True)
mem = report_basic(df, "CA_2024H2 LOC (all shards concatenated)")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["GEO", "LOC_FSA", "LOC_PROVINCE", "LOC_CITY", "POPULATION"])

# regression check: shard 0 alone should still be 352,256 rows
shard0 = pd.read_parquet(shard_paths[0])
check_row_count(shard0.shape[0], 352256, "LOC shard 0 rows (prior finding)")
print(f"NEW observation: total rows across all 5 shards = {df.shape[0]:,}")

# GEO here should NOT be unique (it's a dimension table keyed by geo area,
# not a person-level id) -- confirm that expectation explicitly.
dup_geo = df["GEO"].duplicated().sum()
print(f"duplicate GEO rows: {dup_geo} (expected: 0, since GEO should be this table's key)")


local shards found: 5 (expected 5)
--- CA_2024H2 LOC (all shards concatenated) ---
shape: 873,747 rows x 15 cols
memory footprint: 532.7 MB
dtype counts:
  str: 12
  object: 3
null-only columns: 0
constant (single-value) columns: 0
sample values (5 rows) for ['GEO', 'LOC_FSA', 'LOC_PROVINCE', 'LOC_CITY', 'POPULATION']:
      GEO LOC_FSA LOC_PROVINCE          LOC_CITY POPULATION
0  A0A4B0     A0A           NL    (NL) TREPASSEY        509
1  A0A3R0     A0A           NL    (NL) ST SHOTTS         91
2  A0A3A0     A0A           NL       (NL) MOBILE        406
3  A0A4A0     A0A           NL    (NL) TORS COVE        966
4  A0A1P0     A0A           NL  (NL) CAPE BROYLE        742
[OK] row count — LOC shard 0 rows (prior finding): actual=352,256 expected=352,256
NEW observation: total rows across all 5 shards = 873,747
duplicate GEO rows: 0 (expected: 0, since GEO should be this table's key)


## 6. `CA_2024H2/VARIABLE_MAPPING` (variable dictionary, 1 shard)

Expected: 33,020 rows, columns
`THEME, TABLE_ID, TABLE_NAME, VAR_ID, DESCRIPTION, VAR_VALUES, NATIONAL_COUNT, VALUE_ID`.

In [12]:
shard_paths = sorted(glob.glob(f"{DATA}/CA_2024H2/VARIABLE_MAPPING/*.parquet"))
print(f"local shards found: {len(shard_paths)} (expected 1)")
df = pd.read_parquet(shard_paths[0])
mem = report_basic(df, "CA_2024H2 VARIABLE_MAPPING")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["THEME", "TABLE_ID", "VAR_ID", "DESCRIPTION", "VAR_VALUES"])
check_row_count(df.shape[0], 33020, "VARIABLE_MAPPING rows")


local shards found: 1 (expected 1)
--- CA_2024H2 VARIABLE_MAPPING ---
shape: 33,020 rows x 8 cols
memory footprint: 9.7 MB
dtype counts:
  str: 7
  object: 1
null-only columns: 0
constant (single-value) columns: 0
sample values (5 rows) for ['THEME', 'TABLE_ID', 'VAR_ID', 'DESCRIPTION', 'VAR_VALUES']:
        THEME TABLE_ID    VAR_ID DESCRIPTION VAR_VALUES
0  Demography   vv_dem  vv_dem_1         Age    18 - 19
1  Demography   vv_dem  vv_dem_1         Age    20 - 24
2  Demography   vv_dem  vv_dem_1         Age    25 - 29
3  Demography   vv_dem  vv_dem_1         Age    30 - 34
4  Demography   vv_dem  vv_dem_1         Age    35 - 39
[OK] row count — VARIABLE_MAPPING rows: actual=33,020 expected=33,020


'OK'

## 7. `dem/` (2026 synthetic population, 27 shards)

Expected (from earlier GCS inspection): 33,597,827 total rows across 27
shards, `id` a dense gap-free range 1..33,597,827, columns
`id, geo, vv_dem_1..20`.

In [13]:
shard_paths = sorted(glob.glob(f"{DATA}/dem_2026/*.parquet"))
print(f"local shards found: {len(shard_paths)} (expected 27)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df = pd.concat(dfs, ignore_index=True)
mem = report_basic(df, "2026 dem (all 27 shards concatenated)")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["id", "geo", "vv_dem_1", "vv_dem_2"])
check_row_count(df.shape[0], 33597827, "dem total rows (27 shards)")
check_id_range(df["id"], "dem (2026)", expect_min=1, expect_max=33597827,
               expect_gapfree=True, expect_nunique=33597827)


local shards found: 27 (expected 27)
--- 2026 dem (all 27 shards concatenated) ---
shape: 33,597,827 rows x 22 cols
memory footprint: 5,831.5 MB
dtype counts:
  int64: 21
  str: 1
null-only columns: 0
constant (single-value) columns: 0
sample values (5 rows) for ['id', 'geo', 'vv_dem_1', 'vv_dem_2']:
   id     geo  vv_dem_1  vv_dem_2
0   1  A0A1A0  21440009  21440014
1   2  A0A1A0  21440011  21440014
2   3  A0A1A0  21440008  21440014
3   4  A0A1A0  21440012  21440014
4   5  A0A1A0  21440008  21440014
[OK] row count — dem total rows (27 shards): actual=33,597,827 expected=33,597,827
id check — dem (2026): n=33,597,827 nunique=33,597,827 min=1 max=33597827
  [OK] nunique expected=33,597,827
  [OK] min expected=1
  [OK] max expected=33597827
  [OK] gap-free contiguous range: True


True

## 8. `intact-survey/` (2026 fused client survey, 250 shards — sampled)

Too large to fully cache locally (250 shards, ~32.4M rows total) without
violating the "don't download the entire dataset unnecessarily" project
convention. This section validates a **local sample** (first 3 shards +
the last shard, shard indices 0, 1, 2, 249) for schema/content, and
separately re-derives the *full* row count via a remote metadata-only scan
(parquet footers only, no row data transferred) to regression-check against
the 32,433,918 figure found earlier this session.

The full gap-free `id` range check (0..32,433,917, verified across all 250
shards) was already performed earlier this session (background task
`b0ql4e9oj`); it is not re-derived here from the 4-shard sample, since a
non-contiguous subset of shards would not be expected to be gap-free on its
own -- re-running that full check would require re-scanning all 250 shards'
`id` columns, which this notebook does further below as an explicit
regression re-check.

In [14]:
shard_paths = sorted(glob.glob(f"{DATA}/intact_survey_2026_sample/*.parquet"))
print(f"local sample shards found: {len(shard_paths)} (expected 4: shards 0,1,2,249)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df = pd.concat(dfs, ignore_index=True)
mem = report_basic(df, "2026 intact-survey (4-shard sample)")
null_only, constant = flag_null_or_constant(df)
show_sample_decoded(df, ["id", "geo", "PROV", "age", "SEX2"])

sample_ids = df["id"]
print(f"sample id check: n={len(sample_ids):,} nunique={sample_ids.nunique():,} "
      f"min={sample_ids.min()} max={sample_ids.max()}")
print(f"[{'OK' if sample_ids.is_unique else 'MISMATCH'}] ids unique within sampled shards")


local sample shards found: 4 (expected 4: shards 0,1,2,249)
--- 2026 intact-survey (4-shard sample) ---
shape: 519,145 rows x 212 cols
memory footprint: 986.7 MB
dtype counts:
  int64: 143
  Int64: 48
  object: 15
  float64: 4
  str: 1
  string: 1
null-only columns: 16 -> ['SEX2r96oe', 'Q23B', 'Q23D', 'Q23E', 'Q23F', 'Q23H', 'Q23J', 'Q23I', 'Q25r1', 'Q25r2', 'Q25r3', 'Q25r97', 'Q25r98', 'Q26', 'Q27', 'Q47']
constant (single-value) columns: 3 -> ['noanswerPOSTAL6_rA9A9A9', 'noanswerFOY1_r99', 'noanswerFOY2_r99']
sample values (5 rows) for ['id', 'geo', 'PROV', 'age', 'SEX2']:
    id     geo  PROV  age  SEX2
0  306  A0A1B0    10    5     2
1  404  A0A1B0    10    3     2
2  422  A0A1B0    10    3     2
3  616  A0A1C0    10    2     1
4  679  A0A1C0    10    6     1
sample id check: n=519,145 nunique=519,145 min=239 max=32433895
[OK] ids unique within sampled shards


In [15]:
# Full row-count + full id gap-free regression check via remote metadata scan
# (reads only parquet footers / id columns, not full row data -- consistent
# with the project's "don't download the entire dataset unnecessarily" rule)
import gcsfs
import pyarrow.parquet as pq

fs = gcsfs.GCSFileSystem()
remote_files = sorted(fs.glob(
    "plusco-arima-data-dropzone-prod/ca/2026/intact-survey/*.parquet"))
print(f"remote shards found: {len(remote_files)} (expected 250)")

total_rows = 0
all_ids = []
for p in remote_files:
    with fs.open(p, "rb") as f:
        pf = pq.ParquetFile(f)
        total_rows += pf.metadata.num_rows
        tbl = pf.read(columns=["id"])
    all_ids.extend(tbl.column("id").to_pylist())

check_row_count(total_rows, 32433918, "intact-survey total rows (250 shards, remote metadata)")

id_series = pd.Series(all_ids)
check_id_range(id_series, "intact-survey (2026, full remote re-check)",
               expect_min=0, expect_max=32433917, expect_gapfree=True,
               expect_nunique=32433918)


/Users/wiame.ichane/arima_v24_v26/.venv-1/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


remote shards found: 250 (expected 250)
[OK] row count — intact-survey total rows (250 shards, remote metadata): actual=32,433,918 expected=32,433,918
id check — intact-survey (2026, full remote re-check): n=32,433,918 nunique=32,433,918 min=0 max=32433917
  [OK] nunique expected=32,433,918
  [OK] min expected=0
  [OK] max expected=32433917
  [OK] gap-free contiguous range: True


True

## 9. `variable_mapping.csv` (2026)

**Path correction:** CLAUDE.md documents this as
`gs://plusco-arima-data-dropzone-prod/ca/2026/variable_mapping.csv`, but it
was not found there — it actually lives one level deeper, at
`ca/2026/synthetic-population/variable_mapping.csv`. No prior row-count
baseline exists for this file (first time it's been opened this
engagement), so this section reports fresh observations rather than a
regression check.

In [16]:
path = f"{DATA}/variable_mapping_2026.csv"
df = pd.read_csv(path)
mem = report_basic(df, "2026 variable_mapping")
null_only, constant = flag_null_or_constant(df)
print("columns:", list(df.columns))
show_sample_decoded(df, list(df.columns)[:5])
print(f"NEW observation: {df.shape[0]:,} rows -- no prior baseline to regression-check against.")


--- 2026 variable_mapping ---
shape: 34,234 rows x 9 cols
memory footprint: 6.4 MB
dtype counts:
  str: 7
  int64: 2
null-only columns: 0
constant (single-value) columns: 1 -> ['dataset']
columns: ['dataset', 'theme', 'table_id', 'table_name', 'var_id', 'description', 'var_values', 'national_count', 'value_id']
sample values (5 rows) for ['dataset', 'theme', 'table_id', 'table_name', 'var_id']:
          dataset     theme table_id  table_name       var_id
0             NaN       NaN      All         All          NaN
1  Base Data Pack  Segments      pri  Environics  pri_segment
2  Base Data Pack  Segments      pri  Environics  pri_segment
3  Base Data Pack  Segments      pri  Environics  pri_segment
4  Base Data Pack  Segments      pri  Environics  pri_segment
NEW observation: 34,234 rows -- no prior baseline to regression-check against.


## Summary

Consolidated pass/fail view of every regression check run above. This is a
log of what was checked, not a re-statement of the Part 1.2 findings
themselves (see `outputs/part1_vintage_comparison.md` for those).

In [17]:
print("See per-section [OK]/[MISMATCH] tags above for the full detail.")
print("No corrections were made to any file in this notebook -- report only.")


See per-section [OK]/[MISMATCH] tags above for the full detail.
No corrections were made to any file in this notebook -- report only.
